In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

In [ ]:
# Choose method 2 Phase or Full fine tuning

TWO_PHASE = True        # 2 Phase
# TWO_PHASE = False     # full layer training

In [ ]:
DATASET_DIR = Path("Datasets/RQD").resolve()

MODEL_WEIGHTS = "src/yolo11x-cls.pt"

EPOCHS = 50
PATIENCE = 10

IMGSZ = 360 # or 224
DEVICE = 0 # or "cpu"


RANDOM_SEED = 42

if TWO_PHASE:
    MODE = "2-Phase"
    PHASE1_NAME = "RQD_YOLO11x_phase1_frozen"
    PHASE2_NAME = "RQD_YOLO11x_phase2_finetune"
else:
    MODE = "Full-fine tuning"
    PHASE2_NAME = "RQD_YOLO11x_full-fine_tuning"

PROJECT = Path.cwd() / f"ModelTraining/RQD_YOLO11_{IMGSZ}_{MODE}"
TEST_NAME   = "RQD_YOLO11x_test"

In [ ]:
print(PROJECT)

In [ ]:
common_args = {
    "data": str(DATASET_DIR),
    "imgsz": IMGSZ,
    "batch": 32,
    "device": DEVICE,

    "patience": PATIENCE,

    "seed": RANDOM_SEED,
    "deterministic": True,

    "plots": True,
    "verbose": True,

    "project": PROJECT,
    "exist_ok": True,
}

In [ ]:
if TWO_PHASE:
    print("\n" + "=" * 70)
    print("PHASE 1 - FROZEN YOLO11 BACKBONE")
    print("=" * 70)

    model = YOLO(MODEL_WEIGHTS)


    # YOLO11 classification:
    #
    # Layers 0-9  = backbone
    # Layer 10    = Classify head
    #
    # freeze=10 freezes layers 0-9.


    results_phase1 = model.train(
        **common_args,
        epochs=EPOCHS,
        name=PHASE1_NAME,
        freeze=10
    )

    phase1_dir = Path(model.trainer.save_dir)

    phase1_best = phase1_dir / "weights" / "best.pt"

    print("\nPhase 1 directory:")
    print(phase1_dir)

    print("\nBest Phase 1 model:")
    print(phase1_best)


    if not phase1_best.exists():
        raise FileNotFoundError(
            f"Phase 1 best model was not found: {phase1_best}"
        )

In [ ]:
print("\n" + "=" * 70)

if TWO_PHASE:
    print("PHASE 2 - UNFREEZE AND FINE-TUNING YOLO11")    
    model = YOLO(str(phase1_best))
else:
    print("FULL FINE-TUNING YOLO11")
    model = YOLO(MODEL_WEIGHTS)

print("=" * 70)



results_phase2 = model.train(
    **common_args,
    epochs=EPOCHS,
    name=PHASE2_NAME,

    # No layers frozen.
    freeze=None
)


phase2_dir = Path(model.trainer.save_dir)

phase2_best = phase2_dir / "weights" / "best.pt"


print("\nPhase 2 directory:")
print(phase2_dir)

print("\nBest Phase 2 model:")
print(phase2_best)


if not phase2_best.exists():
    raise FileNotFoundError(
        f"Phase 2 best model was not found: {phase2_best}"
    )

In [ ]:
print("\n" + "=" * 70)
print("FINAL TEST SET EVALUATION")
print("=" * 70)


# Load only the best fine-tuned model
best_model = YOLO(str(phase2_best))


test_metrics = best_model.val(
    data=str(DATASET_DIR),

    split="test",

    imgsz=IMGSZ,
    batch=32,
    device=DEVICE,

    project=PROJECT,
    name=TEST_NAME,

    plots=True
)


print("\n" + "=" * 70)
print("FINAL YOLO11 TEST RESULTS")
print("=" * 70)

print(
    f"Top-1 Test Accuracy: "
    f"{test_metrics.top1:.4f} "
    f"({test_metrics.top1 * 100:.2f}%)"
)

print(
    f"Top-5 Test Accuracy: "
    f"{test_metrics.top5:.4f} "
    f"({test_metrics.top5 * 100:.2f}%)"
)

In [ ]:
if TWO_PHASE:
    phase1_csv = phase1_dir / "results.csv"
    df1 = pd.read_csv(phase1_csv)

    # Remove possible whitespace from column names
    df1.columns = df1.columns.str.strip()

    print("\nPhase 1 columns:")
    print(df1.columns.tolist())


phase2_csv = phase2_dir / "results.csv"

df2 = pd.read_csv(phase2_csv)


# Remove possible whitespace from column names
df2.columns = df2.columns.str.strip()

print("\nPhase 2 columns:")
print(df2.columns.tolist())

In [ ]:
if TWO_PHASE:
    val_acc = (
        df1["metrics/accuracy_top1"].tolist()
        + df2["metrics/accuracy_top1"].tolist()
    )

    train_loss = (
        df1["train/loss"].tolist()
        + df2["train/loss"].tolist()
    )

    val_loss = (
        df1["val/loss"].tolist()
        + df2["val/loss"].tolist()
    )

    phase1_length = len(df1)

else:
    val_acc = df2["metrics/accuracy_top1"].tolist()
    train_loss = df2["train/loss"].tolist()
    val_loss = df2["val/loss"].tolist()


epochs = range(1, len(val_acc) + 1)

In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    val_acc,
    label="Validation Top-1 Accuracy"
)

if TWO_PHASE:
    plt.axvline(
        x=phase1_length,
        linestyle="--",
        label="Start Fine-tuning"
    )

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(f"YOLO11x Validation Accuracy - {IMGSZ}x{IMGSZ} - {MODE}")

plt.legend()

plt.xlim(left=0)

plt.tight_layout()

plt.show()


In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    train_loss,
    label="Train Loss"
)

plt.plot(
    epochs,
    val_loss,
    label="Validation Loss"
)

if TWO_PHASE:
    plt.axvline(
        x=phase1_length,
        linestyle="--",
        label="Start Fine-tuning"
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(f"YOLO11x Loss - {IMGSZ}x{IMGSZ} - {MODE}")

plt.xlim(left=0)

plt.legend()

plt.tight_layout()

plt.show()